# Đề tài 13 – Multi-Delivery Robot: huấn luyện trên Google Colab

Notebook này chạy **toàn bộ** thí nghiệm của project (Q-learning, SARSA, Monte Carlo ES trên 4 map cấp 1 → 4), vẽ lại hình và lưu kết quả.

- **Runtime:** chọn *CPU* là đủ (Runtime → Change runtime type → CPU). Học tăng cường dạng bảng không dùng GPU.
- **Tốc độ đến từ `numba`:** vòng lặp huấn luyện được biên dịch ra mã máy, nhanh hơn bản Python thuần hàng trăm lần.
- Colab miễn phí thường chỉ có **2 nhân CPU**. Ước tính một lượt đầy đủ (bước 3 + 5) mất khoảng 30–45 phút, còn khoảng 15–25 phút nếu đặt `BASELINE = False`. Trên máy 12 nhân, cùng việc đó đo được khoảng 5 phút (221 s + 72 s), nên Colab chủ yếu giúp máy của bạn được rảnh, không nhanh hơn.

**Cách lấy code (chọn một ở ô dưới):**
1. `drive` *(khuyên dùng)*: chép cả thư mục `Project_RL` lên Google Drive, vào `MyDrive/Project_RL`. Kết quả ghi thẳng vào Drive, tắt Colab cũng không mất.
2. `github`: clone từ GitHub (cần push code mới lên trước). Cuối notebook có ô nén kết quả để tải về.

In [ ]:
SOURCE = "drive"                                   # "drive" hoặc "github"
DRIVE_DIR = "/content/drive/MyDrive/Project_RL"    # thư mục project trên Drive
GITHUB_URL = "https://github.com/Qwup2112/multi-delivery-robot-rl.git"

import os
if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = DRIVE_DIR
else:
    WORK = "/content/Project_RL"
    if not os.path.exists(WORK):
        !git clone -q {GITHUB_URL} {WORK}
os.chdir(WORK)
os.environ["NUMBA_CACHE_DIR"] = "/content/numba_cache"   # cache biên dịch để ngoài Drive cho nhanh
print("Thư mục làm việc:", os.getcwd())
!ls

## 1. Cài thư viện và kiểm tra CPU

In [ ]:
!pip install -q gymnasium numba
import numba, multiprocessing
print("numba", numba.__version__, "| số nhân CPU:", multiprocessing.cpu_count())

## 2. Chạy thử nhanh (~1 phút)

2 seed, 1/20 số episode. Chỉ để kiểm tra mọi thứ chạy được; độ chính xác ở bước này còn thấp là bình thường.

In [ ]:
!python experiments.py --quick --no-baseline

## 3. Huấn luyện đầy đủ

- `SCALE`: nhân số episode của mọi map. `1` = số episode trong `configs/*.json` (S1 100k, S2 300k, S3 2M, S4 6M). Muốn học lâu hơn thì đặt `2`, `3`, ...
- `BASELINE`: chạy thêm cấu hình cũ (trước cải tiến) để vẽ hình so sánh `ablation.png`. Đặt `False` cho nhanh gấp đôi.

Log được ghi vào `results/log.txt`.

In [ ]:
SCALE = 1
BASELINE = True

flags = f"--scale {SCALE} --workers {multiprocessing.cpu_count()}" + ("" if BASELINE else " --no-baseline")
!mkdir -p results && python experiments.py {flags} 2>&1 | tee results/log.txt

## 4. Vẽ hình (tiếng Việt → `figures/`, tiếng Anh → `figures_en/`)

In [ ]:
!python plots.py
!python plots.py --lang en

## 5. Đo hội tụ của bảng Q về Q* (Value Iteration)

In [ ]:
!python convergence.py --workers {multiprocessing.cpu_count()}
!python convergence.py --plot-only --lang en

## 6. Demo một episode + ảnh động

In [ ]:
!python demo.py --gif
!python demo.py --gif --lang en

## 7. Xem nhanh kết quả

In [ ]:
import json
from IPython.display import Image, display

s = json.load(open("results/summary.json", encoding="utf-8"))
print("Độ chính xác cuối (% ô xuất phát đi đúng lộ trình tối ưu, trung bình các seed):")
for name, st in s["settings"].items():
    row = [f"{k}: {100 * a['accuracy_mean']:5.1f}%" for g in ("algos", "baseline") for k, a in st.get(g, {}).items()]
    print(f"  {name:22s}", " | ".join(row))

for f in ["maps.png", "accuracy.png", "ablation.png", "learning_curves.png", "convergence.png"]:
    if os.path.exists(f"figures/{f}"):
        display(Image(f"figures/{f}", width=900))

## 8. Tải kết quả về máy (khi dùng `github`)

Khi dùng `drive`, kết quả đã nằm sẵn trong `MyDrive/Project_RL/results` và `figures/`, không cần bước này.

In [ ]:
!zip -qr /content/ket_qua.zip results figures figures_en
from google.colab import files
files.download("/content/ket_qua.zip")